In [1]:
import numpy as np


def random_unitary(dim):
  h = (
      np.random.randn(dim, dim) + 1j * np.random.randn(dim, dim)
  ) / np.sqrt(2)
  q, r = np.linalg.qr(h)
  r_diag = np.diagonal(r)
  ph = r_diag / np.abs(r_diag)
  return q @ np.diag(ph)


def exact_trajectory_operator_evolution(
    num_qubits=4, num_steps=5, measurement_prob=0.3
):
  dim = 2**num_qubits
  # Initialize operator as the Identity matrix
  operator = np.eye(dim, dtype=complex)

  print(
      f"Initial Exact Trajectory Operator Norm (Spectral):"
      f" {np.linalg.norm(operator, 2):.4f}"
  )

  for step in range(num_steps):
    # Apply a random unitary gate layer
    u = random_unitary(dim)
    operator = u @ operator @ u.conj().T

    # Simulate an exact measurement decision point
    if np.random.rand() < measurement_prob:
      q_idx = np.random.randint(0, num_qubits)

      # Construct exact projection operators (Kraus operators)
      proj_0 = np.kron(
          np.kron(np.eye(2**q_idx), np.array([[1, 0], [0, 0]])),
          np.eye(2 ** (num_qubits - q_idx - 1)),
      )
      proj_1 = np.kron(
          np.kron(np.eye(2**q_idx), np.array([[0, 0], [0, 1]])),
          np.eye(2 ** (num_qubits - q_idx - 1)),
      )

      # Exact determination: evaluate branch weights/probabilities via trace
      weight_0 = np.real(np.trace(proj_0 @ operator @ proj_0.conj().T))
      weight_1 = np.real(np.trace(proj_1 @ operator @ proj_1.conj().T))

      # Choose the exact deterministic branch (e.g., the higher probability outcome)
      if weight_0 >= weight_1:
        operator = proj_0 @ operator @ proj_0.conj().T
        action_desc = "Exact Measurement (Outcome 0 selected)"
      else:
        operator = proj_1 @ operator @ proj_1.conj().T
        action_desc = "Exact Measurement (Outcome 1 selected)"
    else:
      action_desc = "Unitary only"

    # Check the spectral norm of the operator along this exact trajectory branch
    norm_val = np.linalg.norm(operator, 2)
    print(f"Step {step + 1}: Norm = {norm_val:.4f} ({action_desc})")


# Run the simulation for an exact trajectory path
np.random.seed(42)
exact_trajectory_operator_evolution()

Initial Exact Trajectory Operator Norm (Spectral): 1.0000
Step 1: Norm = 1.0000 (Exact Measurement (Outcome 0 selected))
Step 2: Norm = 0.9793 (Exact Measurement (Outcome 1 selected))
Step 3: Norm = 0.9793 (Unitary only)
Step 4: Norm = 0.9793 (Unitary only)
Step 5: Norm = 0.8551 (Exact Measurement (Outcome 1 selected))


In [2]:
"""
Local Plotting Script for Weak-Measurement Expressibility Scaling Study.
Reads `expressibility_weak_s_results.pkl` and plots:
  Row 1: Expressibility epsilon_U^(rho) vs p across N (log and linear scale)
  Row 2: Frame Potential F_U^(rho) vs p across N (with dashed Haar bounds)
"""

import os
import argparse
import pickle
import numpy as np
import matplotlib.pyplot as plt


def plot_expressibility_results(pkl_path, out_dir="plots_expressibility", log_scale=True):
    if not os.path.exists(pkl_path):
        raise FileNotFoundError(f"Could not find pickle file at: {pkl_path}")

    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    os.makedirs(out_dir, exist_ok=True)

    N_list = sorted(data.get("N_list", []))
    s_list = sorted(data.get("s_list", []), reverse=True)
    ansatz_list = data.get("ansatz_list", ["HEA1", "HEA2", "HVA"])
    depth_desc = data.get("depth_desc", "")
    samples = data.get("samples", 0)

    expressibilities = data.get("expressibilities", {})
    frame_potentials = data.get("frame_potentials", {})
    haar_potentials = data.get("haar_frame_potentials", {})

    cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(N_list)))
    markers = ["o", "s", "^", "D", "v", "p", "*"]

    for s_val in s_list:
        if s_val not in expressibilities:
            continue

        active_ansatzes = [
            a for a in ansatz_list
            if a in expressibilities[s_val] and len(expressibilities[s_val][a]) > 0
        ]
        if not active_ansatzes:
            continue

        n_cols = len(active_ansatzes)
        fig, axes = plt.subplots(2, n_cols, figsize=(5.5 * n_cols, 8.5), squeeze=False)

        for col_idx, ansatz in enumerate(active_ansatzes):
            ax_eps = axes[0, col_idx]
            ax_fp = axes[1, col_idx]

            p_dict_eps = expressibilities[s_val][ansatz]
            p_dict_fp = frame_potentials[s_val][ansatz]
            p_vals_all = sorted(p_dict_eps.keys())

            for n_idx, N in enumerate(N_list):
                p_valid, eps_valid, fp_valid = [], [], []
                for p in p_vals_all:
                    if N in p_dict_eps[p]:
                        p_valid.append(p)
                        eps_valid.append(p_dict_eps[p][N])
                        fp_valid.append(p_dict_fp[p][N])

                if not p_valid:
                    continue

                color = cmap[n_idx]
                marker = markers[n_idx % len(markers)]

                # Plot 1: Expressibility epsilon_U^(rho)
                eps_arr = np.array(eps_valid, dtype=float)
                if log_scale:
                    # Mask exact zeros from clamping if using log scale
                    eps_plot = np.where(eps_arr > 0, eps_arr, np.nan)
                else:
                    eps_plot = eps_arr

                ax_eps.plot(
                    p_valid, eps_plot, marker=marker, color=color,
                    linewidth=1.8, markersize=5, label=f"$N = {N}$"
                )

                # Plot 2: Frame Potential F_U^(rho) + Haar floor
                ax_fp.plot(
                    p_valid, fp_valid, marker=marker, color=color,
                    linewidth=1.8, markersize=5, label=f"$N = {N}$"
                )
                if N in haar_potentials:
                    ax_fp.axhline(
                        haar_potentials[N], color=color, linestyle="--",
                        linewidth=1.0, alpha=0.55
                    )

            ax_eps.set_title(f"{ansatz} ($s = {s_val:.2f}$, {depth_desc})", fontsize=12, fontweight="bold")
            ax_eps.set_xlabel("Measurement Rate $p$", fontsize=11)
            ax_eps.set_ylabel(r"Expressibility $\varepsilon_{\mathbb{U}}^{\rho}$", fontsize=11)
            if log_scale:
                ax_eps.set_yscale("log")
            ax_eps.grid(True, which="both", linestyle=":", alpha=0.6)
            ax_eps.legend(fontsize=9, loc="best")

            ax_fp.set_title(f"{ansatz} Frame Potential vs Haar Limit", fontsize=11)
            ax_fp.set_xlabel("Measurement Rate $p$", fontsize=11)
            ax_fp.set_ylabel(r"Frame Potential $\mathcal{F}_{\mathbb{U}}^{(\rho)}$", fontsize=11)
            ax_fp.set_yscale("log")
            ax_fp.grid(True, which="both", linestyle=":", alpha=0.6)

        fig.suptitle(
            f"Weak-Measurement Expressibility Scaling ($s = {s_val:.2f}$, $K = {samples}$ Trajectories)",
            fontsize=14, fontweight="bold", y=0.98
        )
        plt.tight_layout(rect=[0, 0, 1, 0.95])

        scale_tag = "log" if log_scale else "linear"
        out_file = os.path.join(out_dir, f"expressibility_s{s_val:.2f}_{scale_tag}.png")
        plt.savefig(out_file, dpi=300)
        plt.close(fig)
        print(f"Saved figure -> {out_file}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--pkl", type=str,
        default="expressibility_weak_s_results.pkl",
        help="Path to expressibility_weak_s_results.pkl"
    )
    parser.add_argument("--out-dir", type=str, default="plots_expressibility")
    parser.add_argument("--linear", action="store_true", help="Plot expressibility on a linear y-axis instead of log")
    # args = parser.parse_args()
    args, unknown = parser.parse_known_args()
    plot_expressibility_results(args.pkl, out_dir=args.out_dir, log_scale=(not args.linear))

Saved figure -> plots_expressibility\expressibility_s1.00_log.png


In [3]:
"""
Local Plotting Script for Weak-Measurement Expressibility Scaling Study.
Reads `expressibility_weak_s_results.pkl` and plots:
  1. Expressibility epsilon_U^(rho) vs p across N (log and linear scale)
  2. Frame Potential F_U^(rho) vs p across N (with dashed Haar bounds)
These are saved as separate figures.
"""

import os
import argparse
import pickle
import numpy as np
import matplotlib.pyplot as plt


def plot_expressibility_results(pkl_path, out_dir="plots_expressibility", log_scale=True):
    if not os.path.exists(pkl_path):
        raise FileNotFoundError(f"Could not find pickle file at: {pkl_path}")

    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    os.makedirs(out_dir, exist_ok=True)

    N_list = sorted(data.get("N_list", []))
    s_list = sorted(data.get("s_list", []), reverse=True)
    ansatz_list = data.get("ansatz_list", ["HEA1", "HEA2", "HVA"])
    depth_desc = data.get("depth_desc", "")
    samples = data.get("samples", 0)

    expressibilities = data.get("expressibilities", {})
    frame_potentials = data.get("frame_potentials", {})
    haar_potentials = data.get("haar_frame_potentials", {})

    cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(N_list)))
    markers = ["o", "s", "^", "D", "v", "p", "*"]

    for s_val in s_list:
        if s_val not in expressibilities:
            continue

        active_ansatzes = [
            a for a in ansatz_list
            if a in expressibilities[s_val] and len(expressibilities[s_val][a]) > 0
        ]
        if not active_ansatzes:
            continue

        n_cols = len(active_ansatzes)
        
        # Create separate figures for Expressibility and Frame Potential
        fig_eps, axes_eps = plt.subplots(1, n_cols, figsize=(5.5 * n_cols, 5), squeeze=False)
        fig_fp, axes_fp = plt.subplots(1, n_cols, figsize=(5.5 * n_cols, 5), squeeze=False)

        for col_idx, ansatz in enumerate(active_ansatzes):
            ax_eps = axes_eps[0, col_idx]
            ax_fp = axes_fp[0, col_idx]

            p_dict_eps = expressibilities[s_val][ansatz]
            p_dict_fp = frame_potentials[s_val][ansatz]
            p_vals_all = sorted(p_dict_eps.keys())

            for n_idx, N in enumerate(N_list):
                p_valid, eps_valid, fp_valid = [], [], []
                for p in p_vals_all:
                    if N in p_dict_eps[p]:
                        p_valid.append(p)
                        eps_valid.append(p_dict_eps[p][N])
                        fp_valid.append(p_dict_fp[p][N])

                if not p_valid:
                    continue

                color = cmap[n_idx]
                marker = markers[n_idx % len(markers)]

                # Plot 1: Expressibility epsilon_U^(rho)
                eps_arr = np.array(eps_valid, dtype=float)
                if log_scale:
                    # Mask exact zeros from clamping if using log scale
                    eps_plot = np.where(eps_arr > 0, eps_arr, np.nan)
                else:
                    eps_plot = eps_arr

                ax_eps.plot(
                    p_valid, eps_plot, marker=marker, color=color,
                    linewidth=1.8, markersize=5, label=f"$N = {N}$"
                )

                # Plot 2: Frame Potential F_U^(rho) + Haar floor
                ax_fp.plot(
                    p_valid, fp_valid, marker=marker, color=color,
                    linewidth=1.8, markersize=5, label=f"$N = {N}$"
                )
                if N in haar_potentials:
                    ax_fp.axhline(
                        haar_potentials[N], color=color, linestyle="--",
                        linewidth=1.0, alpha=0.55
                    )

            # Format Expressibility Axes
            ax_eps.set_title(f"{ansatz} ($s = {s_val:.2f}$, {depth_desc})", fontsize=12, fontweight="bold")
            ax_eps.set_xlabel("Measurement Rate $p$", fontsize=11)
            ax_eps.set_ylabel(r"Expressibility $\varepsilon_{\mathbb{U}}^{\rho}$", fontsize=11)
            if log_scale:
                ax_eps.set_yscale("log")
            ax_eps.grid(True, which="both", linestyle=":", alpha=0.6)
            ax_eps.legend(fontsize=9, loc="best")

            # Format Frame Potential Axes
            ax_fp.set_title(f"{ansatz} Frame Potential vs Haar Limit", fontsize=11)
            ax_fp.set_xlabel("Measurement Rate $p$", fontsize=11)
            ax_fp.set_ylabel(r"Frame Potential $\mathcal{F}_{\mathbb{U}}^{(\rho)}$", fontsize=11)
            ax_fp.set_yscale("log")
            ax_fp.grid(True, which="both", linestyle=":", alpha=0.6)
            ax_fp.legend(fontsize=9, loc="best")

        # Save Expressibility Figure
        fig_eps.suptitle(
            f"Weak-Measurement Expressibility Scaling ($s = {s_val:.2f}$, $K = {samples}$ Trajectories)",
            fontsize=14, fontweight="bold", y=0.98
        )
        fig_eps.tight_layout(rect=[0, 0, 1, 0.95])
        scale_tag = "log" if log_scale else "linear"
        out_file_eps = os.path.join(out_dir, f"expressibility_eps_s{s_val:.2f}_{scale_tag}.png")
        fig_eps.savefig(out_file_eps, dpi=300)
        plt.close(fig_eps)
        print(f"Saved Expressibility figure -> {out_file_eps}")

        # Save Frame Potential Figure
        fig_fp.suptitle(
            f"Weak-Measurement Frame Potential ($s = {s_val:.2f}$, $K = {samples}$ Trajectories)",
            fontsize=14, fontweight="bold", y=0.98
        )
        fig_fp.tight_layout(rect=[0, 0, 1, 0.95])
        out_file_fp = os.path.join(out_dir, f"frame_potential_s{s_val:.2f}.png")
        fig_fp.savefig(out_file_fp, dpi=300)
        plt.close(fig_fp)
        print(f"Saved Frame Potential figure -> {out_file_fp}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--pkl", type=str,
        default="expressibility_weak_s_results.pkl",
        help="Path to expressibility_weak_s_results.pkl"
    )
    parser.add_argument("--out-dir", type=str, default="plots_expressibility")
    parser.add_argument("--linear", action="store_true", help="Plot expressibility on a linear y-axis instead of log")
    args, unknown = parser.parse_known_args()
    plot_expressibility_results(args.pkl, out_dir=args.out_dir, log_scale=(not args.linear))

Saved Expressibility figure -> plots_expressibility\expressibility_eps_s1.00_log.png
Saved Frame Potential figure -> plots_expressibility\frame_potential_s1.00.png
